In [21]:
import numpy as np
import matplotlib.pyplot as plt
import stim
import pymatching
import sinter
from typing import List
import time
import pandas as pd
from IPython.display import HTML
import lmfit
import itertools
from lmfit import Model
from stimbposd import SinterDecoder_BPOSD, sinter_decoders
from stimbposd import BPOSD
from itertools import combinations
import galois
import ldpc
from ldpc import bposd_decoder

In [50]:
h=3   ###Height of grid  (NOW WORKS)
w=25  ###Width of grid, make odd k=w-4
H=[]
stabs=[]
packed=[]

for i in range(1,h):
    for j in range(0,w):          
        stab=[]
        stab.append([[i,j], [i-1,j-1],[i-1,j],[i-1,j+1]])

        sta=np.zeros([h,w])
        for pack in stab[0]:
            if pack[1] >= 0 and pack[1] < w:
                sta[pack[0]][pack[1]]=1

        sta = sta[::-1]

        rows=[]
        for r in range(h):
            if r == 0:
                rows.append(sta[r])
            else:
                rows.append(sta[r][r:-r])

        final_sta = np.concatenate(rows).tolist()

        if np.sum(final_sta) > 0:
            H.append(final_sta)

H=np.array(H).astype(int)

In [51]:
def gf2_rref(A):
    A = A.copy()
    m, n = A.shape
    row = 0
    pivots = []

    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col] == 1:
                pivot = r
                break
        if pivot is None:
            continue

        A[[row, pivot]] = A[[pivot, row]]
        pivots.append(col)

        for r in range(m):
            if r != row and A[r, col] == 1:
                A[r] ^= A[row]

        row += 1
        if row == m:
            break

    return A, pivots

def nullspace_gf2(H):
    R, pivots = gf2_rref(H)
    n = H.shape[1]
    free = [i for i in range(n) if i not in pivots]

    G = []
    for f in free:
        v = np.zeros(n, dtype=int)
        v[f] = 1
        for i, p in enumerate(pivots):
            v[p] = R[i, f]
        G.append(v)

    return np.array(G)

G_LDPC = nullspace_gf2(H)   # 12 × 23

# -----------------------------
# Convert G to systematic form
# -----------------------------
def systematic_form(G):
    G = G.copy()
    rows, cols = G.shape
    r = 0
    col_perm = list(range(cols))

    for c in range(cols):
        if r >= rows:
            break
        for rr in range(r, rows):
            if G[rr, c]:
                G[[r, rr]] = G[[rr, r]]
                col_perm[r], col_perm[c] = col_perm[c], col_perm[r]
                G[:, [r, c]] = G[:, [c, r]]
                for rrr in range(rows):
                    if rrr != r and G[rrr, r]:
                        G[rrr] ^= G[r]
                r += 1
                break

    return G, col_perm

#G_sys, perm = systematic_form(G)

print(len(G_LDPC))



21


In [52]:
import ldpc.codes
import ldpc.code_util
d, number_code_words_sampled, lowest_weight_codewords = ldpc.code_util.estimate_code_distance(np.array(H), timeout_seconds = 15)
print(f"Code distance estimate, d <= {d} (no. codewords sampled: {number_code_words_sampled})")

Code distance estimate, d <= 7 (no. codewords sampled: 25081890)


In [53]:
code_supports = []
for j in range(G_LDPC.shape[1]):      # loop over columns
    rows = np.where(G_LDPC[:, j] == 1)[0]
    code_supports.append([int(r % 3) for r in rows])

print(code_supports)

[[0], [1], [0, 2], [1, 0], [0, 2, 1], [1, 0, 2], [2, 1, 0], [0, 2, 1], [1, 0, 2], [2, 1, 0], [0, 2, 1], [1, 0, 2], [2, 1, 0], [0, 2, 1], [1, 0, 2], [2, 1, 0], [0, 2, 1], [1, 0, 2], [2, 1, 0], [0, 2, 1], [1, 0, 2], [2, 1], [0, 2], [1], [2], [0], [0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2, 0], [2, 0, 1], [0, 1, 2], [1, 2], [2], [0], [1], [2], [0], [1], [2], [0], [1], [2], [0], [1], [2], [0], [1], [2], [0], [1], [2], [0], [1], [2]]


In [54]:
def all_unique(lst):
    return len(lst) == len(set(lst))

for i in range(len(code_supports)):
    if not all_unique(code_supports[i]):
        print("Failure at qubit", i)

# Pointy stabilizer dz = 7, h = 3, has a partition number of 3

In [55]:
print(len(H[0]))

69


# Will concatenate a [69,21,dz=7] code into 3 x [7,4,3] codes for an overall [69, 12, (dz=7, dx=3)] code

In [57]:
Hd = [list(np.where(row == 1)[0]) for row in H]
Hd = [[int(x) for x in stab] for stab in Hd]
print(Hd)

[[25, 48], [26, 48, 49], [27, 48, 49, 50], [28, 49, 50, 51], [29, 50, 51, 52], [30, 51, 52, 53], [31, 52, 53, 54], [32, 53, 54, 55], [33, 54, 55, 56], [34, 55, 56, 57], [35, 56, 57, 58], [36, 57, 58, 59], [37, 58, 59, 60], [38, 59, 60, 61], [39, 60, 61, 62], [40, 61, 62, 63], [41, 62, 63, 64], [42, 63, 64, 65], [43, 64, 65, 66], [44, 65, 66, 67], [45, 66, 67, 68], [46, 67, 68], [47, 68], [0, 25], [1, 25, 26], [2, 25, 26, 27], [3, 26, 27, 28], [4, 27, 28, 29], [5, 28, 29, 30], [6, 29, 30, 31], [7, 30, 31, 32], [8, 31, 32, 33], [9, 32, 33, 34], [10, 33, 34, 35], [11, 34, 35, 36], [12, 35, 36, 37], [13, 36, 37, 38], [14, 37, 38, 39], [15, 38, 39, 40], [16, 39, 40, 41], [17, 40, 41, 42], [18, 41, 42, 43], [19, 42, 43, 44], [20, 43, 44, 45], [21, 44, 45, 46], [22, 45, 46, 47], [23, 46, 47], [24, 47]]


In [56]:
H_outer= [
    [1, 0, 1, 0, 1, 0, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1]]

G_outer = nullspace_gf2(np.array(H_outer))

In [58]:
G_1=[]
G_2=[]
G_3=[]
for i in range(7):
    G_1.append(G_LDPC[3*i])
    G_2.append(G_LDPC[3*i+1])
    G_3.append(G_LDPC[3*i+2])

In [59]:
n = 69
m = n - 1  # number of checks

H_rep = np.zeros((m, n), dtype=int)

for i in range(m):
    H_rep[i, i] = 1
    H_rep[i, i+1] = 1

print(H_rep)

[[1 1 0 ... 0 0 0]
 [0 1 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 1 1 0]
 [0 0 0 ... 0 1 1]]


In [60]:
H_outer=np.array(H_outer)
Outer_1=(H_outer @ G_1) %2
Outer_2=(H_outer @ G_2) %2
Outer_3=(H_outer @ G_3) %2

In [61]:
def sparse_to_dense(H, n_cols=None, dtype=np.uint8):
    n_rows = len(H)

    if n_cols is None:
        n_cols = max(max(row) for row in H) + 1

    M = np.zeros((n_rows, n_cols), dtype=dtype)

    for i, row in enumerate(H):
        M[i, row] = 1

    return M

In [62]:

def rep_code_syns(cnots,H_LDPC, H_REP):  #Presuming all inputs are dense
 
    decoder = bposd_decoder(
        H_REP.T,
        error_rate=0.01,     # physical bit-flip probability
        max_iter=1000,         # BP iterations
        bp_method="ms", 
        osd_method ="osd_cs",     # "ms" = min-sum (default)
        osd_order=0          # OSD order (0 = BP only)
    )
    outputs=[]
    for i in range(len(H_LDPC)):
        syndrome = H_LDPC[i] * cnots
        decoding = decoder.decode(syndrome) 
        c=np.flatnonzero(decoding)
        outputs.append([c])
    outputs = [arr.tolist() for [arr] in outputs]
    return(outputs)


print(rep_code_syns(Outer_1[0], H, H_rep))

[[25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47], [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47], [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47], [], [], [], [31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53], [32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53], [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53], [], [], [], [37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59], [38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59], [39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59], [], [], [], [43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65], [44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58

c:\Users\shanahpe\anaconda3\envs\qec_env\Lib\site-packages\ldpc\_legacy_ldpc_v1\_legacy_bposd_decoder.py:45: UserWarning: This is the old syntax for the `bposd_decoder` from `ldpc v1`. Use the `BpOsdDecoder` class from `ldpc v2` for additional features.
  warnings.warn(


In [65]:
print(G_outer)
Short_X=[]
for i in G_outer:
    op_1=[]
    op_2=[]
    op_3=[]
    for j in i:
        op_1.append(int(j))
        op_1.append(0)
        op_1.append(0)

        op_2.append(0)
        op_2.append(int(j))
        op_2.append(0)

        op_3.append(0)
        op_3.append(0)
        op_3.append(int(j))


    Short_X.append(op_1)
    Short_X.append(op_2)
    Short_X.append(op_3)

for i in range(len(Short_X)):
    Short_X[i]= [0]*48+Short_X[i]

Short_X = [np.array(row) for row in Short_X]

X_log_d = [list(np.where(row == 1)[0]) for row in Short_X]

print(X_log_d)

[[1 1 1 0 0 0 0]
 [1 0 0 1 1 0 0]
 [0 1 0 1 0 1 0]
 [1 1 0 1 0 0 1]]
[[np.int64(48), np.int64(51), np.int64(54)], [np.int64(49), np.int64(52), np.int64(55)], [np.int64(50), np.int64(53), np.int64(56)], [np.int64(48), np.int64(57), np.int64(60)], [np.int64(49), np.int64(58), np.int64(61)], [np.int64(50), np.int64(59), np.int64(62)], [np.int64(51), np.int64(57), np.int64(63)], [np.int64(52), np.int64(58), np.int64(64)], [np.int64(53), np.int64(59), np.int64(65)], [np.int64(48), np.int64(51), np.int64(57), np.int64(66)], [np.int64(49), np.int64(52), np.int64(58), np.int64(67)], [np.int64(50), np.int64(53), np.int64(59), np.int64(68)]]


In [66]:
o1q0=([rep_code_syns(Outer_1[0], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_1[1], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_1[2], [Short_X[0]], H_rep)])
o1q0 = [x for sublist in o1q0 for x in sublist]
o2q0=([rep_code_syns(Outer_2[0], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_2[1], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_2[2], [Short_X[0]], H_rep)])
o2q0 = [x for sublist in o2q0 for x in sublist]
o3q0=([rep_code_syns(Outer_3[0], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_3[1], [Short_X[0]], H_rep)]+[rep_code_syns(Outer_3[2], [Short_X[0]], H_rep)])
o3q0 = [x for sublist in o3q0 for x in sublist]

print(Short_X[0])
print(o1q0[0])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[48, 49, 50, 51, 52, 53]


In [67]:
print(Outer_1[0])

[1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 0 1 1 1 0 0 0
 1 1 1 0 0 0 1 1 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0]


In [68]:
def c_m(x,y,N):
    return(y*N+x)

In [88]:
def circ(rnds):
        circuit=stim.Circuit()  
        measurement_count=0

        for i in range(0,69):
                circuit.append("Qubit_COORDS",c_m(i,0,69),[0,i])  #Memory Layer Data
        for i in range(0,48):
                circuit.append("Qubit_COORDS",c_m(i,1,69),[1,i])  #Memory Layer Ancilla

        for i in range(0,69):
                circuit.append("Qubit_COORDS",c_m(i,2,69),[2,i]) #Comp Layer Data 1
        for i in range(0,68):
                circuit.append("Qubit_COORDS",c_m(i,3,69),[3,i]) #Comp Layer Ancilla 1

        for i in range(0,69):
                circuit.append("Qubit_COORDS",c_m(i,4,69),[4,i]) #Comp Layer Data 2
        for i in range(0,68):
                circuit.append("Qubit_COORDS",c_m(i,5,69),[5,i]) #Comp Layer Ancilla 2

        for i in range(0,69):
                circuit.append("Qubit_COORDS",c_m(i,6,69),[6,i]) #Comp Layer Data 3
        for i in range(0,68):
                circuit.append("Qubit_COORDS",c_m(i,7,69),[7,i]) #Comp Layer Ancilla 3

        for i in range(0,69):
                circuit.append("Rx",c_m(i,0,69))            #Memory Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,2,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,4,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,6,69))            #Comp Data Initialised

        circuit.append("Tick")

        for k in range(rnds):

                for i in range(0,48):
                        circuit.append("Rx",c_m(i,1,69)) 


                for i in range(0,68):
                        circuit.append("Rx",c_m(i,3,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,5,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,7,69))  

                circuit.append("Tick") 

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 3:
                                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 4:
                                circuit.append("CX",[c_m(i,1,96),c_m(Hd[i][3],0,69)])

                circuit.append("Tick")


                ldpc_meas=measurement_count
                for i in range(48):                           ##LDPC Code Auxiliary Measurements
                        circuit.append("Mx",c_m(i,1,69))
                        measurement_count+=1
                

                if k==0:
                        gap_0a=measurement_count
                if k == 1:
                        gap_0b=measurement_count
                        gap_0 = gap_0b-gap_0a


                code_1_meas=measurement_count
                o1c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,3,69))
                        measurement_count+=1
  
                code_2_meas=measurement_count
                o2c0_meas=measurement_count
                for i in range(48):                          
                        circuit.append("Mx",c_m(i,5,69))
                        measurement_count+=1

                code_3_meas=measurement_count
                o3c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,7,69))
                        measurement_count+=1

        for i in range(69):                                                 #######CNOTs of the first check for all 3 codes 
                if Outer_1[0][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,2,69)])
                if Outer_2[0][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,4,69)])
                if Outer_3[0][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,6,69)])


        for i in range(0,48):
                circuit.append("Rx",c_m(i,1,69)) 


        for i in range(0,68):
                circuit.append("Rx",c_m(i,3,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,5,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,7,69))  

        circuit.append("Tick") 

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 3:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 4:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][3],0,69)])

        circuit.append("Tick")

        fin_c_1=measurement_count
        for i in range(48):                           ##LDPC Code Auxiliary Measurements
                circuit.append("Mx",c_m(i,1,69))
                measurement_count+=1

        backprob_indexes_1=rep_code_syns(Outer_1[0],  H, H_rep)
        backprob_indexes_2=rep_code_syns(Outer_2[0],  H, H_rep)
        backprob_indexes_3=rep_code_syns(Outer_3[0],  H, H_rep)



        final_r_1=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,3,69))
                measurement_count+=1
 
        final_r_2=measurement_count
        for i in range(68):                          
                circuit.append("Mx",c_m(i,5,69))
                measurement_count+=1

        final_r_3=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,7,69))
                measurement_count+=1


        for i in range(69):
                circuit.append("Mz",c_m(i,2,69))        
                measurement_count +=1


        for i in range(69):
                circuit.append("Mz",c_m(i,4,69))        
                measurement_count +=1

        for i in range(69):
                circuit.append("Mz",c_m(i,6,69))        #####################################################First Check Complete, all 3 codes
                measurement_count +=1




        for i in range(0,69):
                circuit.append("Rz",c_m(i,2,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,4,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,6,69))            #Comp Data Initialised

        circuit.append("Tick")

        for k in range(rnds):

                for i in range(0,48):
                        circuit.append("Rx",c_m(i,1,69)) 


                for i in range(0,68):
                        circuit.append("Rx",c_m(i,3,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,5,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,7,69))  

                circuit.append("Tick") 

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 3:
                                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 4:
                                circuit.append("CX",[c_m(i,1,96),c_m(Hd[i][3],0,69)])

                circuit.append("Tick")


                ldpc_meas=measurement_count
                for i in range(48):                           ##LDPC Code Auxiliary Measurements
                        circuit.append("Mx",c_m(i,1,69))
                        measurement_count+=1
                

                if k==0:
                        gap_0a=measurement_count
                if k == 1:
                        gap_0b=measurement_count
                        gap_0 = gap_0b-gap_0a


                code_1_meas=measurement_count
                o1c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,3,69))
                        measurement_count+=1
 
                
                code_2_meas=measurement_count
                o2c0_meas=measurement_count
                for i in range(48):                          
                        circuit.append("Mx",c_m(i,5,69))
                        measurement_count+=1

                code_3_meas=measurement_count
                o3c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,7,69))
                        measurement_count+=1

        for i in range(69):                                                 #######CNOTs of the first check for all 3 codes 
                if Outer_1[1][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,2,69)])
                if Outer_2[1][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,4,69)])
                if Outer_3[1][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,6,69)])


        for i in range(0,48):
                circuit.append("Rx",c_m(i,1,69)) 


        for i in range(0,68):
                circuit.append("Rx",c_m(i,3,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,5,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,7,69))  

        circuit.append("Tick") 

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 3:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 4:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][3],0,69)])

        circuit.append("Tick")

        fin_c_1=measurement_count
        for i in range(48):                           ##LDPC Code Auxiliary Measurements
                circuit.append("Mx",c_m(i,1,69))
                measurement_count+=1

        backprob_indexes_1=rep_code_syns(Outer_1[1],  H, H_rep)
        backprob_indexes_2=rep_code_syns(Outer_2[1],  H, H_rep)
        backprob_indexes_3=rep_code_syns(Outer_3[1],  H, H_rep)



        final_r_1=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,3,69))
                measurement_count+=1
 
        final_r_2=measurement_count
        for i in range(68):                          
                circuit.append("Mx",c_m(i,5,69))
                measurement_count+=1

        final_r_3=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,7,69))
                measurement_count+=1


        for i in range(69):
                circuit.append("Mz",c_m(i,2,69))        
                measurement_count +=1


        for i in range(69):
                circuit.append("Mz",c_m(i,4,69))        
                measurement_count +=1

        for i in range(69):
                circuit.append("Mz",c_m(i,6,69))        #####################################################Second Check Complete, all 3 codes
                measurement_count +=1


        for i in range(0,69):
                circuit.append("Rz",c_m(i,2,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,4,69))            #Comp Data Initialised

        for i in range(0,69):
                circuit.append("Rz",c_m(i,6,69))            #Comp Data Initialised

        circuit.append("Tick")

        for k in range(rnds):

                for i in range(0,48):
                        circuit.append("Rx",c_m(i,1,69)) 


                for i in range(0,68):
                        circuit.append("Rx",c_m(i,3,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,5,69))  

                for i in range(0,68):
                        circuit.append("Rx",c_m(i,7,69))  

                circuit.append("Tick") 

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

                for i in range(68):
                        circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
                for i in range(68):
                        circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 3:
                                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

                circuit.append("Tick")

                for i in range(len(Hd)):
                        if len(Hd[i]) >= 4:
                                circuit.append("CX",[c_m(i,1,96),c_m(Hd[i][3],0,69)])

                circuit.append("Tick")


                ldpc_meas=measurement_count
                for i in range(48):                           ##LDPC Code Auxiliary Measurements
                        circuit.append("Mx",c_m(i,1,69))
                        measurement_count+=1
                

                if k==0:
                        gap_0a=measurement_count
                if k == 1:
                        gap_0b=measurement_count
                        gap_0 = gap_0b-gap_0a


                code_1_meas=measurement_count
                o1c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,3,69))
                        measurement_count+=1
           
                code_2_meas=measurement_count
                o2c0_meas=measurement_count
                for i in range(48):                          
                        circuit.append("Mx",c_m(i,5,69))
                        measurement_count+=1

                code_3_meas=measurement_count
                o3c0_meas=measurement_count
                for i in range(68):                           
                        circuit.append("Mx",c_m(i,7,69))
                        measurement_count+=1

        for i in range(69):                                                 #######CNOTs of the first check for all 3 codes 
                if Outer_1[2][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,2,69)])
                if Outer_2[2][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,4,69)])
                if Outer_3[2][i]==1:
                        circuit.append("CX",[c_m(i,0,69),c_m(i,6,69)])


        for i in range(0,48):
                circuit.append("Rx",c_m(i,1,69)) 


        for i in range(0,68):
                circuit.append("Rx",c_m(i,3,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,5,69))  

        for i in range(0,68):
                circuit.append("Rx",c_m(i,7,69))  

        circuit.append("Tick") 

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][0],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][1],0,69)])

        for i in range(68):
                circuit.append("CX",[c_m(i,3,69),c_m(i+1,2,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,5,69),c_m(i+1,4,69)])
        for i in range(68):
                circuit.append("CX",[c_m(i,7,69),c_m(i+1,6,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 3:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][2],0,69)])

        circuit.append("Tick")

        for i in range(len(Hd)):
                if len(Hd[i]) >= 4:
                        circuit.append("CX",[c_m(i,1,69),c_m(Hd[i][3],0,69)])

        circuit.append("Tick")

        fin_c_1=measurement_count
        for i in range(48):                           ##LDPC Code Auxiliary Measurements
                circuit.append("Mx",c_m(i,1,69))
                measurement_count+=1

        backprob_indexes_1=rep_code_syns(Outer_1[2],  H, H_rep)
        backprob_indexes_2=rep_code_syns(Outer_2[2],  H, H_rep)
        backprob_indexes_3=rep_code_syns(Outer_3[2],  H, H_rep)



        final_r_1=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,3,69))
                measurement_count+=1
 
        final_r_2=measurement_count
        for i in range(68):                          
                circuit.append("Mx",c_m(i,5,69))
                measurement_count+=1

        final_r_3=measurement_count
        for i in range(68):                           
                circuit.append("Mx",c_m(i,7,69))
                measurement_count+=1


        for i in range(69):
                circuit.append("Mz",c_m(i,2,69))        
                measurement_count +=1


        for i in range(69):
                circuit.append("Mz",c_m(i,4,69))        
                measurement_count +=1

        for i in range(69):
                circuit.append("Mz",c_m(i,6,69))        #####################################################Third Check Complete, all 3 codes
                measurement_count +=1


        for i in range(69):
                circuit.append("Mx",c_m(i,0,69))          ######Final Measurement on data qubits
                measurement_count +=1
        

        return(circuit)

In [89]:
html_content = f"""
<div style="overflow-x: auto; overflow-y: hidden;">
    <div style="white-space: nowrap;width: {100}%;">{circ(1).diagram('timeline-svg')}</div>
</div>
"""
display(HTML(html_content))